# 🏗️ SNOW Data Product — Setup & Infrastructure
## Domain: IT Service Management | Catalog: `snow_product` | Owner: ITSM Team

This notebook creates the complete infrastructure for the **SNOW Data Product** — a self-contained, independently deployable data product following Data Mesh principles.

**Self-Contained Data Product Characteristics:**
| Characteristic | Implementation |
|---|---|
| ✅ **Discoverable** | Tags on catalog, schemas, tables; rich COMMENT metadata |
| ✅ **Addressable** | `snow_product.gold.service_health` — fully qualified, unique |
| ✅ **Trustworthy** | Quality checks, data contracts, SLA monitoring |
| ✅ **Self-Describing** | Every column has a COMMENT; TBLPROPERTIES with owner, domain |
| ✅ **Interoperable** | Delta format, standard schemas, Delta Sharing |
| ✅ **Secure** | CDF for audit, UDFs for column masking & row filtering |

> **This domain operates independently** — no dependency on the Data Mesh Hub.

## Step 1: Create Catalog & Schemas
The ITSM team owns the entire `snow_product` catalog. Schemas follow **Medallion Architecture** (Bronze → Silver → Gold) plus a **Governance** schema for self-contained quality, contracts, and observability.

In [ ]:
%sql
-- ============================================================
-- DOMAIN CATALOG: snow_product (already created at metastore level)
-- If running for the first time, create the catalog from the
-- metastore admin workspace or Account Console.
-- ============================================================
USE CATALOG snow_product;

-- Medallion schemas
CREATE SCHEMA IF NOT EXISTS snow_product.bronze  COMMENT 'Raw Zone | Ingested from ServiceNow | Owner: ITSM Team';
CREATE SCHEMA IF NOT EXISTS snow_product.silver  COMMENT 'Curated Zone | Cleansed and enriched | Owner: ITSM Team';
CREATE SCHEMA IF NOT EXISTS snow_product.gold    COMMENT 'Product Zone | Business-ready data products | Owner: ITSM Team';

-- Governance schema (self-contained — no hub dependency)
CREATE SCHEMA IF NOT EXISTS snow_product.governance COMMENT 'Governance Zone | Quality, contracts, observability, registry | Owner: ITSM Team';

## Step 2: Create Bronze Tables
Raw ingestion targets for ServiceNow data. Preserves original field values.

In [ ]:
%sql
-- Bronze: Raw incidents from ServiceNow
CREATE OR REPLACE TABLE snow_product.bronze.raw_incidents (
  incident_id STRING COMMENT 'ServiceNow incident number (INC-XXX)',
  short_description STRING COMMENT 'Brief incident summary',
  priority INT COMMENT 'Priority level: 1=Critical, 2=High, 3=Medium, 4=Low',
  state STRING COMMENT 'Current state: New, In Progress, On Hold, Resolved',
  category STRING COMMENT 'Incident category: Access, Performance, Software, etc.',
  assigned_to STRING COMMENT 'Assignee name',
  assignment_group STRING COMMENT 'Support group handling the incident',
  affected_application_id STRING COMMENT 'FK to application registry (APP-XXX)',
  created_at TIMESTAMP COMMENT 'Incident creation timestamp',
  resolved_at TIMESTAMP COMMENT 'Resolution timestamp (NULL if open)',
  sla_breached BOOLEAN COMMENT 'Whether SLA was breached'
) COMMENT 'Raw incidents ingested from ServiceNow Table API | Source: incident table'
TBLPROPERTIES ('domain' = 'ITSM', 'team_owner' = 'ITSM Team', 'layer' = 'bronze', 'source_system' = 'ServiceNow');

-- Bronze: Raw change requests from ServiceNow  
CREATE OR REPLACE TABLE snow_product.bronze.raw_change_requests (
  change_id STRING COMMENT 'ServiceNow change number (CHG-XXX)',
  description STRING COMMENT 'Change request description',
  change_type STRING COMMENT 'Type: Normal, Standard, Emergency',
  risk STRING COMMENT 'Risk assessment: Low, Medium, High',
  state STRING COMMENT 'Current state: Scheduled, Closed, Cancelled',
  affected_application_id STRING COMMENT 'FK to application registry (APP-XXX)',
  requested_by STRING COMMENT 'Requester name',
  planned_start TIMESTAMP COMMENT 'Planned start of change window',
  planned_end TIMESTAMP COMMENT 'Planned end of change window',
  success BOOLEAN COMMENT 'Whether change was successful (NULL if pending)'
) COMMENT 'Raw change requests ingested from ServiceNow Table API | Source: change_request table'
TBLPROPERTIES ('domain' = 'ITSM', 'team_owner' = 'ITSM Team', 'layer' = 'bronze', 'source_system' = 'ServiceNow');

## Step 3: Create Silver Tables
Curated tables with derived fields, standardized values, and validation.

In [ ]:
%sql
-- Silver: Enriched incidents with derived metrics
CREATE OR REPLACE TABLE snow_product.silver.incidents (
  incident_id STRING, short_description STRING, priority INT, priority_label STRING,
  state STRING, category STRING, assigned_to STRING, assignment_group STRING,
  affected_application_id STRING, created_at TIMESTAMP, resolved_at TIMESTAMP,
  resolution_hours DOUBLE, sla_breached BOOLEAN, is_open BOOLEAN,
  processed_at TIMESTAMP
) COMMENT 'Curated incidents — enriched with priority labels, resolution hours, open/closed flags'
TBLPROPERTIES ('domain' = 'ITSM', 'team_owner' = 'ITSM Team', 'layer' = 'silver');

-- Silver: Enriched change requests with duration metrics
CREATE OR REPLACE TABLE snow_product.silver.change_requests (
  change_id STRING, description STRING, change_type STRING, risk STRING,
  state STRING, affected_application_id STRING, requested_by STRING,
  planned_start TIMESTAMP, planned_end TIMESTAMP, success BOOLEAN,
  duration_hours DOUBLE, processed_at TIMESTAMP
) COMMENT 'Curated change requests — enriched with duration calculations'
TBLPROPERTIES ('domain' = 'ITSM', 'team_owner' = 'ITSM Team', 'layer' = 'silver');

## Step 4: Create Gold Table — Service Health Data Product
The **Service Health** data product aggregates incidents and changes per application. This is what IT Operations and management consume for SLA reviews.

In [ ]:
%sql
-- Gold: Service Health — the domain's primary data product
CREATE OR REPLACE TABLE snow_product.gold.service_health (
  affected_application_id STRING COMMENT 'Application identifier (APP-XXX)',
  total_incidents INT COMMENT 'Total incident count for this application',
  open_incidents INT COMMENT 'Currently open incidents',
  critical_incidents INT COMMENT 'Priority 1 (Critical) incidents',
  high_incidents INT COMMENT 'Priority 2 (High) incidents',
  sla_breaches INT COMMENT 'Number of SLA breaches',
  sla_compliance_pct DOUBLE COMMENT 'SLA compliance percentage (0-100)',
  avg_resolution_hours DOUBLE COMMENT 'Average time to resolve incidents (hours)',
  total_changes INT COMMENT 'Total change requests for this application',
  change_success_rate_pct DOUBLE COMMENT 'Percentage of successful changes (0-100)',
  risk_score INT COMMENT 'Composite risk score (lower = safer)',
  product_generated_at TIMESTAMP COMMENT 'Timestamp when this product was last refreshed'
) COMMENT 'Service Health per application — aggregated from incidents + change requests | SLA: 2h freshness | Quality: >90%'
TBLPROPERTIES (
  'domain' = 'ITSM', 'team_owner' = 'ITSM Team', 'layer' = 'gold',
  'data_product' = 'true', 'sla_freshness_hours' = '2',
  'quality_threshold_pct' = '90', 'delta.enableChangeDataFeed' = 'true'
);

## Step 5: Create Governance Tables (Self-Contained)
Each domain owns its own governance — no hub dependency required.

In [ ]:
%sql
-- Data Product Registry — tracks this domain's products
CREATE OR REPLACE TABLE snow_product.governance.data_product_registry (
  product_id STRING, product_name STRING, domain STRING, owner_group STRING,
  owner_contact STRING, table_name STRING, status STRING,
  sla_freshness_hours INT, quality_threshold_pct DOUBLE,
  quality_score DOUBLE, freshness_status STRING, row_count LONG,
  schema_version STRING, consumers INT, created_at TIMESTAMP, updated_at TIMESTAMP,
  share_name STRING, listing_status STRING
) COMMENT 'Registry of SNOW domain data products with lifecycle and distribution status';

-- Data Contracts — agreements with consumers
CREATE OR REPLACE TABLE snow_product.governance.data_contracts (
  contract_id STRING, product_name STRING, producer_group STRING,
  consumer_group STRING, contract_status STRING, agreed_sla_hours INT,
  quality_threshold_pct DOUBLE, schema_version STRING,
  retention_days INT, escalation_contact STRING,
  created_at TIMESTAMP, updated_at TIMESTAMP
) COMMENT 'Data contracts between SNOW domain and consumers';

-- Data Quality Log — quality check results
CREATE OR REPLACE TABLE snow_product.governance.data_quality_log (
  check_id STRING, check_timestamp TIMESTAMP, data_product STRING,
  domain STRING, table_name STRING, check_name STRING, check_type STRING,
  expected_value STRING, actual_value STRING, passed BOOLEAN,
  severity STRING, details STRING
) COMMENT 'Quality check results for SNOW domain data products';

-- Product Observability — time-series health monitoring
CREATE OR REPLACE TABLE snow_product.governance.product_observability (
  product_name STRING, check_timestamp TIMESTAMP,
  quality_score DOUBLE, freshness_hours DOUBLE,
  sla_met BOOLEAN, row_count LONG, column_count INT,
  schema_drift BOOLEAN, status STRING
) COMMENT 'Observability metrics for SNOW domain data products';

## Step 6: Tags for Discoverability
Unity Catalog tags at catalog, schema, and table levels make assets discoverable via search and Catalog Explorer.

In [ ]:
%sql
-- Tag catalog with domain ownership
ALTER CATALOG snow_product SET TAGS ('domain' = 'IT Service Management', 'team_owner' = 'ITSM Team', 'data_mesh_role' = 'Domain Catalog');

-- Tag schemas with layer information
ALTER SCHEMA snow_product.bronze     SET TAGS ('domain' = 'ITSM', 'layer' = 'bronze', 'team_owner' = 'ITSM Team');
ALTER SCHEMA snow_product.silver     SET TAGS ('domain' = 'ITSM', 'layer' = 'silver', 'team_owner' = 'ITSM Team');
ALTER SCHEMA snow_product.gold       SET TAGS ('domain' = 'ITSM', 'layer' = 'gold', 'team_owner' = 'ITSM Team');
ALTER SCHEMA snow_product.governance SET TAGS ('domain' = 'ITSM', 'layer' = 'governance', 'team_owner' = 'ITSM Team');

-- Tag gold data product table
ALTER TABLE snow_product.gold.service_health SET TAGS (
  'data_product' = 'true', 'domain' = 'ITSM', 'pii' = 'false', 
  'classification' = 'internal', 'sla_hours' = '2', 'quality_threshold' = '90'
);

## Step 7: Access Control
The ITSM team has full control. Consumers get read-only access to gold products.

In [ ]:
%sql
-- Read access for all workspace users (for demo purposes)
GRANT USE CATALOG ON CATALOG snow_product TO `account users`;
GRANT USE SCHEMA ON CATALOG snow_product TO `account users`;
GRANT SELECT ON CATALOG snow_product TO `account users`;

## ✅ Setup Complete
Infrastructure is ready:
- **Catalog**: `snow_product` with ITSM domain metadata
- **Schemas**: bronze, silver, gold, governance (4 schemas)
- **Tables**: 2 bronze + 2 silver + 1 gold + 4 governance = **9 tables**
- **Tags**: Catalog, schema, and table level — fully discoverable
- **Grants**: Account users can read; ITSM team owns

Next: Run `01_Snow_Pipeline` to populate the data product.